In [ ]:
#!/usr/bin/env python3
"""
Hallucination Labeler
─────────────────────
A desktop GUI for labeling JSONL records with:
  - human_label: 1 (hallucinated) or 0 (not hallucinated)
  - hallucination_type: one of a fixed set of categories, or a custom one
  - reason: free-text note explaining the label

Labels are written back to the original file in real-time.
Every field (and the whole record) can be copied to the clipboard,
e.g. to paste into an LLM.

Usage:
    python3 hallucination_labeler.py
    python3 hallucination_labeler.py path/to/your_file.jsonl   # open directly
"""

import json
import sys
import os
import tkinter as tk
from tkinter import ttk, filedialog, messagebox, simpledialog
from pathlib import Path
from datetime import datetime


# ── Colour palette ────────────────────────────────────────────────────────────
BG          = "#1e1e2e"
PANEL       = "#2a2a3d"
BORDER      = "#383850"
TEXT        = "#cdd6f4"
MUTED       = "#6c7086"
ACCENT      = "#89b4fa"
GREEN       = "#a6e3a1"
RED         = "#f38ba8"
YELLOW      = "#f9e2af"
WHITE       = "#ffffff"
BTN_BG      = "#313244"
BTN_HOVER   = "#45475a"

FONT_MONO   = ("Courier New", 10)
FONT_LABEL  = ("Segoe UI", 10)
FONT_BOLD   = ("Segoe UI", 10, "bold")
FONT_TITLE  = ("Segoe UI", 13, "bold")
FONT_BIG    = ("Segoe UI", 16, "bold")
FONT_SMALL  = ("Segoe UI", 7)

# ── Hallucination taxonomy ────────────────────────────────────────────────────
HALLUCINATION_TYPES = [
    "unsupported_expansion",
    "wrong_entity",
    "wrong_number",
    "contradiction",
    "fabricated_information",
    "incorrect_procedure",
    "missing_context",
]
ADD_CUSTOM_LABEL = "+ Add custom type…"


# ── Helpers ───────────────────────────────────────────────────────────────────

def load_jsonl(path: str) -> list[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_no}: {e}") from e
    return records


def save_jsonl(path: str, records: list[dict]) -> None:
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    os.replace(tmp, path)   # atomic replace


def fmt_field(value) -> str:
    if value is None:
        return "(none)"
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, indent=2)


# ── Main Application ──────────────────────────────────────────────────────────

class HallucinationLabeler:

    # ── Fields shown in the detail panel (in order) ──────────────────────────
    DETAIL_FIELDS = [
        ("Question",      "question"),
        ("Answer",        "answer"),
        ("Evidence",      "evidence"),
        ("RAG Answer",    "rag_answer"),
        ("Type",          "question_type"),
        ("Difficulty",    "difficulty"),
        ("Source Doc",    "source_doc"),
        ("Chunk ID",      "chunk_id"),
        ("ID",            "id"),
    ]

    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("Hallucination Labeler")
        self.root.configure(bg=BG)
        self.root.minsize(1150, 760)

        self.filepath: str | None = None
        self.records:  list[dict] = []
        self.current:  int        = 0
        self.filter_mode: str     = "all"   # all | unlabeled | hallucinated | not_hallucinated

        self._build_ui()
        self._apply_styles()

        # Open file passed on the command line, if any
        if len(sys.argv) > 1 and Path(sys.argv[1]).is_file():
            self._open_file(sys.argv[1])

    # ── UI construction ───────────────────────────────────────────────────────

    def _build_ui(self):
        # ── Top bar ──────────────────────────────────────────────────────────
        top = tk.Frame(self.root, bg=BG, pady=10, padx=16)
        top.pack(fill="x")

        tk.Label(top, text="Hallucination Labeler", font=FONT_BIG,
                 bg=BG, fg=ACCENT).pack(side="left")

        btn_frame = tk.Frame(top, bg=BG)
        btn_frame.pack(side="right")

        self._mk_btn(btn_frame, "📂  Open JSONL", self._choose_file, ACCENT).pack(side="left", padx=4)
        self._mk_btn(btn_frame, "💾  Save Copy As…", self._save_copy, BTN_BG).pack(side="left", padx=4)

        # ── Status bar (file info) ────────────────────────────────────────────
        self.status_var = tk.StringVar(value="No file loaded — click 'Open JSONL' to start.")
        status_bar = tk.Frame(self.root, bg=PANEL, pady=6, padx=16)
        status_bar.pack(fill="x")
        tk.Label(status_bar, textvariable=self.status_var, font=FONT_LABEL,
                 bg=PANEL, fg=MUTED, anchor="w").pack(side="left", fill="x", expand=True)

        # ── Progress bar ─────────────────────────────────────────────────────
        self.progress_var = tk.DoubleVar(value=0)
        self.progress = ttk.Progressbar(self.root, variable=self.progress_var,
                                        maximum=100, mode="determinate")
        self.progress.pack(fill="x", padx=0, pady=0)

        # ── Main pane (left = list, right = detail) ───────────────────────────
        main = tk.Frame(self.root, bg=BG)
        main.pack(fill="both", expand=True, padx=0)

        # Left panel – record list
        left = tk.Frame(main, bg=PANEL, width=340)
        left.pack(side="left", fill="y")
        left.pack_propagate(False)

        self._build_left(left)

        # Right panel – detail view
        right = tk.Frame(main, bg=BG)
        right.pack(side="left", fill="both", expand=True)

        self._build_right(right)

        # ── Bottom navigation bar ─────────────────────────────────────────────
        nav = tk.Frame(self.root, bg=PANEL, pady=10, padx=16)
        nav.pack(fill="x", side="bottom")

        self._mk_btn(nav, "◀◀  First",    self._go_first,    BTN_BG).pack(side="left", padx=3)
        self._mk_btn(nav, "◀  Prev",      self._go_prev,     BTN_BG).pack(side="left", padx=3)
        self._mk_btn(nav, "Next  ▶",      self._go_next,     BTN_BG).pack(side="left", padx=3)
        self._mk_btn(nav, "Last  ▶▶",     self._go_last,     BTN_BG).pack(side="left", padx=3)

        # Jump-to
        tk.Label(nav, text="Go to #", font=FONT_LABEL, bg=PANEL, fg=MUTED).pack(side="left", padx=(20, 4))
        self.jump_var = tk.StringVar()
        jump_entry = tk.Entry(nav, textvariable=self.jump_var, width=6,
                              bg=BTN_BG, fg=TEXT, insertbackground=TEXT,
                              relief="flat", font=FONT_LABEL)
        jump_entry.pack(side="left")
        jump_entry.bind("<Return>", self._go_jump)

        self.pos_label = tk.Label(nav, text="", font=FONT_BOLD, bg=PANEL, fg=TEXT)
        self.pos_label.pack(side="right", padx=8)

        # Keyboard shortcuts
        self.root.bind("<Left>",  lambda e: self._go_prev())
        self.root.bind("<Right>", lambda e: self._go_next())
        self.root.bind("h",       lambda e: self._label(1))
        self.root.bind("n",       lambda e: self._label(0))
        self.root.bind("u",       lambda e: self._unlabel())
        self.root.bind("<F5>",    lambda e: self._refresh_list())

    def _build_left(self, parent):
        # Filter bar
        filter_bar = tk.Frame(parent, bg=PANEL, pady=8, padx=10)
        filter_bar.pack(fill="x")

        tk.Label(filter_bar, text="Filter:", font=FONT_LABEL, bg=PANEL, fg=MUTED).pack(side="left")

        self.filter_var = tk.StringVar(value="all")
        filters = [("All", "all"), ("Unlabeled", "unlabeled"),
                   ("✓ Not Hallucinated", "not_hallucinated"),
                   ("✗ Hallucinated", "hallucinated")]
        for text, val in filters:
            tk.Radiobutton(filter_bar, text=text, variable=self.filter_var, value=val,
                           command=self._refresh_list,
                           bg=PANEL, fg=TEXT, selectcolor=PANEL,
                           activebackground=PANEL, activeforeground=ACCENT,
                           font=FONT_LABEL).pack(anchor="w", pady=1)

        # Search
        search_frame = tk.Frame(parent, bg=PANEL, padx=10, pady=4)
        search_frame.pack(fill="x")
        self.search_var = tk.StringVar()
        search_entry = tk.Entry(search_frame, textvariable=self.search_var,
                                bg=BTN_BG, fg=MUTED, insertbackground=TEXT, relief="flat",
                                font=FONT_LABEL)
        search_entry.pack(fill="x")
        # Manual placeholder behaviour
        search_entry.insert(0, "Search…")
        # Attach the trace only after the placeholder text has been inserted,
        # so that insertion doesn't fire _refresh_list() before self.listbox exists.
        self.search_var.trace_add("write", lambda *_: self._refresh_list())
        def _on_focus_in(e):
            if search_entry.get() == "Search…":
                search_entry.delete(0, "end")
                search_entry.config(fg=TEXT)
        def _on_focus_out(e):
            if not search_entry.get():
                search_entry.insert(0, "Search…")
                search_entry.config(fg=MUTED)
        search_entry.bind("<FocusIn>",  _on_focus_in)
        search_entry.bind("<FocusOut>", _on_focus_out)

        # List
        list_frame = tk.Frame(parent, bg=PANEL)
        list_frame.pack(fill="both", expand=True)

        scrollbar = ttk.Scrollbar(list_frame, orient="vertical")
        scrollbar.pack(side="right", fill="y")

        self.listbox = tk.Listbox(
            list_frame,
            yscrollcommand=scrollbar.set,
            bg=PANEL, fg=TEXT,
            selectbackground=ACCENT, selectforeground=BG,
            font=FONT_LABEL,
            relief="flat",
            activestyle="none",
            borderwidth=0,
        )
        self.listbox.pack(fill="both", expand=True)
        scrollbar.config(command=self.listbox.yview)
        self.listbox.bind("<<ListboxSelect>>", self._on_list_select)

        # Stats
        self.stats_label = tk.Label(parent, text="", font=FONT_LABEL,
                                    bg=PANEL, fg=MUTED, pady=8)
        self.stats_label.pack()

    def _build_right(self, parent):
        # Label buttons (big, prominent)
        btn_row = tk.Frame(parent, bg=BG, pady=12)
        btn_row.pack(fill="x", padx=16)

        tk.Label(btn_row, text="Label this record:", font=FONT_BOLD,
                 bg=BG, fg=MUTED).pack(side="left", padx=(0, 12))

        self.btn_not_hall = self._mk_btn(
            btn_row, "✓  NOT Hallucinated  (N)",
            lambda: self._label(0), GREEN, fg=BG, padx=18, pady=8, font=FONT_BOLD
        )
        self.btn_not_hall.pack(side="left", padx=6)

        self.btn_hall = self._mk_btn(
            btn_row, "✗  HALLUCINATED  (H)",
            lambda: self._label(1), RED, fg=BG, padx=18, pady=8, font=FONT_BOLD
        )
        self.btn_hall.pack(side="left", padx=6)

        self._mk_btn(btn_row, "⊘  Unlabel  (U)",
                     self._unlabel, BTN_BG, padx=12).pack(side="left", padx=6)

        self.copy_json_btn = self._mk_btn(
            btn_row, "📋  Copy Record JSON", self._copy_record_json, BTN_BG, padx=12
        )
        self.copy_json_btn.pack(side="left", padx=6)

        # Current label badge
        self.badge_var = tk.StringVar(value="")
        self.badge = tk.Label(btn_row, textvariable=self.badge_var,
                              font=FONT_BOLD, bg=BG, fg=MUTED)
        self.badge.pack(side="right", padx=12)

        # ── Hallucination type + reason row ────────────────────────────────
        meta_row = tk.Frame(parent, bg=BG)
        meta_row.pack(fill="x", padx=16, pady=(0, 12))

        tk.Label(meta_row, text="Type:", font=FONT_LABEL, bg=BG, fg=MUTED).pack(side="left")
        self.type_var = tk.StringVar(value="")
        self.type_combo = ttk.Combobox(
            meta_row, textvariable=self.type_var,
            values=HALLUCINATION_TYPES + [ADD_CUSTOM_LABEL],
            state="readonly", width=26, font=FONT_LABEL
        )
        self.type_combo.pack(side="left", padx=(6, 18))
        self.type_combo.bind("<<ComboboxSelected>>", self._on_type_selected)

        tk.Label(meta_row, text="Reason:", font=FONT_LABEL, bg=BG, fg=MUTED).pack(side="left")
        self.reason_var = tk.StringVar()
        self.reason_entry = tk.Entry(
            meta_row, textvariable=self.reason_var,
            bg=BTN_BG, fg=TEXT, insertbackground=TEXT,
            relief="flat", font=FONT_LABEL
        )
        self.reason_entry.pack(side="left", padx=6, fill="x", expand=True)
        self.reason_entry.bind("<FocusOut>", lambda e: self._save_reason())
        self.reason_entry.bind("<Return>",   lambda e: self._save_reason())

        self._mk_btn(meta_row, "Save", self._save_reason, BTN_BG, padx=10).pack(side="left", padx=(6, 0))

        # Divider
        tk.Frame(parent, bg=BORDER, height=1).pack(fill="x", padx=16)

        # Scrollable detail area
        canvas_frame = tk.Frame(parent, bg=BG)
        canvas_frame.pack(fill="both", expand=True, padx=16, pady=8)

        v_scroll = ttk.Scrollbar(canvas_frame, orient="vertical")
        v_scroll.pack(side="right", fill="y")

        self.canvas = tk.Canvas(canvas_frame, bg=BG, yscrollcommand=v_scroll.set,
                                highlightthickness=0)
        self.canvas.pack(fill="both", expand=True)
        v_scroll.config(command=self.canvas.yview)

        self.detail_frame = tk.Frame(self.canvas, bg=BG)
        self.canvas_window = self.canvas.create_window((0, 0), window=self.detail_frame,
                                                        anchor="nw")

        self.detail_frame.bind("<Configure>", self._on_frame_configure)
        self.canvas.bind("<Configure>", self._on_canvas_configure)
        self.canvas.bind("<MouseWheel>",
                         lambda e: self.canvas.yview_scroll(-1 * (e.delta // 120), "units"))
        self.canvas.bind("<Button-4>",
                         lambda e: self.canvas.yview_scroll(-1, "units"))
        self.canvas.bind("<Button-5>",
                         lambda e: self.canvas.yview_scroll(1, "units"))

        # Placeholder
        self.placeholder = tk.Label(
            self.detail_frame,
            text="Open a JSONL file to start labeling.\n\nKeyboard shortcuts:\n"
                 "  H — mark as Hallucinated (human_label = 1)\n"
                 "  N — mark as Not Hallucinated (human_label = 0)\n"
                 "  U — remove label\n  ← / → — previous / next record\n\n"
                 "Each field has its own 📋 Copy button, and there's a\n"
                 "'Copy Record JSON' button to copy the whole record —\n"
                 "handy for pasting into an LLM.",
            font=FONT_LABEL, bg=BG, fg=MUTED, justify="left"
        )
        self.placeholder.pack(padx=20, pady=40, anchor="w")

    # ── Styles ────────────────────────────────────────────────────────────────

    def _apply_styles(self):
        style = ttk.Style()
        style.theme_use("default")
        style.configure("Horizontal.TProgressbar",
                        troughcolor=PANEL, background=ACCENT, thickness=4)
        style.configure("Vertical.TScrollbar",
                        background=BTN_BG, troughcolor=PANEL, arrowcolor=MUTED)
        style.configure("TCombobox",
                        fieldbackground=BTN_BG, background=BTN_BG,
                        foreground=TEXT, arrowcolor=TEXT)
        self.root.option_add("*TCombobox*Listbox.background", BTN_BG)
        self.root.option_add("*TCombobox*Listbox.foreground", TEXT)
        self.root.option_add("*TCombobox*Listbox.selectBackground", ACCENT)
        self.root.option_add("*TCombobox*Listbox.selectForeground", BG)

    # ── Widget helpers ────────────────────────────────────────────────────────

    def _mk_btn(self, parent, text, cmd, bg=BTN_BG, fg=TEXT,
                padx=10, pady=5, font=FONT_LABEL):
        btn = tk.Button(
            parent, text=text, command=cmd,
            bg=bg, fg=fg, activebackground=BTN_HOVER, activeforeground=WHITE,
            relief="flat", cursor="hand2",
            font=font, padx=padx, pady=pady,
        )
        btn.bind("<Enter>", lambda e: btn.config(bg=self._lighten(bg)))
        btn.bind("<Leave>", lambda e: btn.config(bg=bg))
        return btn

    @staticmethod
    def _lighten(hex_color: str) -> str:
        """Return a slightly lighter version of a hex colour."""
        try:
            r = int(hex_color[1:3], 16)
            g = int(hex_color[3:5], 16)
            b = int(hex_color[5:7], 16)
            r = min(255, r + 25)
            g = min(255, g + 25)
            b = min(255, b + 25)
            return f"#{r:02x}{g:02x}{b:02x}"
        except Exception:
            return hex_color

    # ── Clipboard helpers ────────────────────────────────────────────────────

    def _copy_to_clipboard(self, text: str):
        self.root.clipboard_clear()
        self.root.clipboard_append(text)
        self.root.update()   # keep it on the clipboard after focus changes

    def _copy_record_json(self):
        if not self.records:
            return
        idx = self._filtered_index()
        if idx is None:
            return
        rec = self.records[idx]
        text = json.dumps(rec, indent=2, ensure_ascii=False)
        self._copy_to_clipboard(text)

        # brief visual feedback on the button
        original = self.copy_json_btn.cget("text")
        self.copy_json_btn.config(text="✅  Copied!")
        self.root.after(1000, lambda: self.copy_json_btn.config(text=original))

    # ── File I/O ──────────────────────────────────────────────────────────────

    def _choose_file(self):
        path = filedialog.askopenfilename(
            title="Open JSONL file",
            filetypes=[("JSONL files", "*.jsonl *.ndjson"), ("All files", "*.*")]
        )
        if path:
            self._open_file(path)

    def _open_file(self, path: str):
        try:
            records = load_jsonl(path)
        except Exception as e:
            messagebox.showerror("Error loading file", str(e))
            return

        self.filepath = path
        self.records  = records
        self.current  = 0
        self._refresh_list()
        self._update_status()

        if records:
            self._show_record(self._filtered_index())

    def _save_copy(self):
        if not self.records:
            messagebox.showinfo("Nothing to save", "Load a file first.")
            return
        path = filedialog.asksaveasfilename(
            title="Save labelled copy",
            defaultextension=".jsonl",
            filetypes=[("JSONL", "*.jsonl")]
        )
        if path:
            try:
                save_jsonl(path, self.records)
                messagebox.showinfo("Saved", f"Saved {len(self.records)} records to:\n{path}")
            except Exception as e:
                messagebox.showerror("Save error", str(e))

    def _write_to_file(self):
        """Persist records back to the original file."""
        if self.filepath:
            try:
                save_jsonl(self.filepath, self.records)
            except Exception as e:
                messagebox.showerror("Write error",
                                     f"Could not save to original file:\n{e}")

    # ── Labeling ──────────────────────────────────────────────────────────────

    def _label(self, human_label: int):
        """human_label: 1 = hallucinated, 0 = not hallucinated."""
        if not self.records:
            return
        idx = self._filtered_index()
        if idx is None:
            return
        self.records[idx]["human_label"] = human_label
        self._write_to_file()
        self._refresh_list()
        self._show_record(self._filtered_index())
        self._update_status()
        # Auto-advance to next unlabeled
        self._advance()

    def _unlabel(self):
        if not self.records:
            return
        idx = self._filtered_index()
        if idx is None:
            return
        self.records[idx].pop("human_label", None)
        self.records[idx].pop("hallucination_type", None)
        self.records[idx].pop("reason", None)
        self._write_to_file()
        self._refresh_list()
        self._show_record(self._filtered_index())
        self._update_status()

    def _on_type_selected(self, _event=None):
        val = self.type_var.get()
        if val == ADD_CUSTOM_LABEL:
            custom = simpledialog.askstring(
                "Custom hallucination type",
                "Enter a custom hallucination type:",
                parent=self.root
            )
            custom = (custom or "").strip()
            if custom:
                values = list(self.type_combo["values"])
                if custom not in values:
                    # keep the "add custom" entry last
                    values.insert(len(values) - 1, custom)
                    self.type_combo["values"] = values
                self.type_var.set(custom)
                self._save_type()
            else:
                # Revert to whatever was saved before, if anything
                idx = self._filtered_index()
                current = self.records[idx].get("hallucination_type", "") if idx is not None else ""
                self.type_var.set(current)
            return
        self._save_type()

    def _save_type(self):
        if not self.records:
            return
        idx = self._filtered_index()
        if idx is None:
            return
        val = self.type_var.get().strip()
        if val and val != ADD_CUSTOM_LABEL:
            self.records[idx]["hallucination_type"] = val
        else:
            self.records[idx].pop("hallucination_type", None)
        self._write_to_file()
        self._refresh_list()

    def _save_reason(self):
        if not self.records:
            return
        idx = self._filtered_index()
        if idx is None:
            return
        val = self.reason_var.get().strip()
        if val:
            self.records[idx]["reason"] = val
        else:
            self.records[idx].pop("reason", None)
        self._write_to_file()

    def _advance(self):
        """Move to next record in current view."""
        filtered = self._get_filtered()
        if self.current < len(filtered) - 1:
            self.current += 1
            self._sync_listbox()
            self._show_record(self._filtered_index())

    # ── Navigation ────────────────────────────────────────────────────────────

    def _go_next(self):
        filtered = self._get_filtered()
        if self.current < len(filtered) - 1:
            self.current += 1
            self._sync_listbox()
            self._show_record(self._filtered_index())

    def _go_prev(self):
        if self.current > 0:
            self.current -= 1
            self._sync_listbox()
            self._show_record(self._filtered_index())

    def _go_first(self):
        if self.records:
            self.current = 0
            self._sync_listbox()
            self._show_record(self._filtered_index())

    def _go_last(self):
        filtered = self._get_filtered()
        if filtered:
            self.current = len(filtered) - 1
            self._sync_listbox()
            self._show_record(self._filtered_index())

    def _go_jump(self, _event=None):
        try:
            n = int(self.jump_var.get()) - 1
            filtered = self._get_filtered()
            if 0 <= n < len(filtered):
                self.current = n
                self._sync_listbox()
                self._show_record(self._filtered_index())
        except ValueError:
            pass

    def _on_list_select(self, _event=None):
        sel = self.listbox.curselection()
        if sel:
            self.current = sel[0]
            self._show_record(self._filtered_index())

    def _sync_listbox(self):
        self.listbox.selection_clear(0, "end")
        self.listbox.selection_set(self.current)
        self.listbox.see(self.current)

    # ── Filtering ─────────────────────────────────────────────────────────────

    def _get_filtered(self) -> list[tuple[int, dict]]:
        """Return list of (original_index, record) matching current filter+search."""
        mode   = self.filter_var.get()
        search = self.search_var.get().lower().strip()
        result = []
        for i, rec in enumerate(self.records):
            # Filter by label state
            if mode == "unlabeled" and "human_label" in rec:
                continue
            if mode == "hallucinated" and rec.get("human_label") != 1:
                continue
            if mode == "not_hallucinated" and rec.get("human_label") != 0:
                continue
            # Search filter
            if search and search != "search…":
                haystack = " ".join(str(v) for v in rec.values()).lower()
                if search not in haystack:
                    continue
            result.append((i, rec))
        return result

    def _filtered_index(self) -> int | None:
        """Return the original records index for self.current in filtered view."""
        filtered = self._get_filtered()
        if not filtered or self.current >= len(filtered):
            return None
        return filtered[self.current][0]

    def _refresh_list(self):
        if not hasattr(self, "listbox"):
            return
        filtered = self._get_filtered()
        self.listbox.delete(0, "end")
        for pos, (_, rec) in enumerate(filtered):
            hall = rec.get("human_label")
            prefix = "✓ " if hall == 0 else "✗ " if hall == 1 else "· "
            q = rec.get("question", rec.get("id", "?"))
            short = (q[:55] + "…") if len(q) > 55 else q
            self.listbox.insert("end", f"{prefix}{short}")
            # Colour rows
            if hall == 1:
                self.listbox.itemconfig(pos, fg=RED)
            elif hall == 0:
                self.listbox.itemconfig(pos, fg=GREEN)
            else:
                self.listbox.itemconfig(pos, fg=TEXT)

        # Clamp current
        if filtered:
            self.current = min(self.current, len(filtered) - 1)
        else:
            self.current = 0

        self._sync_listbox()
        self._update_stats(filtered)

        # Update position label
        total = len(filtered)
        pos   = self.current + 1 if filtered else 0
        self.pos_label.config(text=f"{pos} / {total}")

        # Progress
        total_all    = len(self.records)
        labeled      = sum(1 for r in self.records if "human_label" in r)
        pct          = (labeled / total_all * 100) if total_all else 0
        self.progress_var.set(pct)

    def _update_stats(self, filtered):
        total    = len(self.records)
        labeled  = sum(1 for r in self.records if "human_label" in r)
        hall     = sum(1 for r in self.records if r.get("human_label") == 1)
        not_hall = sum(1 for r in self.records if r.get("human_label") == 0)
        unlabeled= total - labeled
        pct      = f"{labeled/total*100:.1f}%" if total else "0%"
        self.stats_label.config(
            text=f"Total: {total}  |  Labeled: {labeled} ({pct})\n"
                 f"✗ Hallucinated: {hall}   ✓ Not: {not_hall}   · Unlabeled: {unlabeled}"
        )

    # ── Detail view ───────────────────────────────────────────────────────────

    def _show_record(self, orig_idx: int | None):
        # Clear detail frame
        for w in self.detail_frame.winfo_children():
            w.destroy()

        if orig_idx is None or not self.records:
            tk.Label(self.detail_frame, text="No records to display.",
                     font=FONT_LABEL, bg=BG, fg=MUTED).pack(padx=20, pady=20)
            self.type_var.set("")
            self.reason_var.set("")
            return

        rec = self.records[orig_idx]

        # Update badge
        hall = rec.get("human_label")
        if hall == 1:
            self.badge_var.set("Current: ✗ HALLUCINATED  (human_label = 1)")
            self.badge.config(fg=RED)
        elif hall == 0:
            self.badge_var.set("Current: ✓ NOT HALLUCINATED  (human_label = 0)")
            self.badge.config(fg=GREEN)
        else:
            self.badge_var.set("Current: · Unlabeled")
            self.badge.config(fg=MUTED)

        # Sync type/reason widgets with this record
        rec_type = rec.get("hallucination_type", "") or ""
        if rec_type:
            values = list(self.type_combo["values"])
            if rec_type not in values:
                values.insert(len(values) - 1, rec_type)
                self.type_combo["values"] = values
        self.type_var.set(rec_type)
        self.reason_var.set(rec.get("reason", "") or "")

        # Render each field
        for label, key in self.DETAIL_FIELDS:
            value = rec.get(key)
            if value is None:
                continue
            self._render_field(self.detail_frame, label, fmt_field(value), key)

        # Show any extra unknown fields
        known_keys = ({k for _, k in self.DETAIL_FIELDS}
                      | {"human_label", "hallucination_type", "reason"})
        for key, value in rec.items():
            if key not in known_keys:
                self._render_field(self.detail_frame, key, fmt_field(value), key)

        # Reset scroll
        self.canvas.yview_moveto(0)

        # Update position
        filtered = self._get_filtered()
        total    = len(filtered)
        pos      = self.current + 1 if filtered else 0
        self.pos_label.config(text=f"{pos} / {total}")

    def _render_field(self, parent, label: str, value: str, key: str):
        colours = {
            "question":      ACCENT,
            "answer":        GREEN,
            "rag_answer":    YELLOW,
            "evidence":      "#cba6f7",   # mauve
        }
        accent_col = colours.get(key, TEXT)

        container = tk.Frame(parent, bg=BG, pady=6)
        container.pack(fill="x", padx=20, anchor="w")

        header = tk.Frame(container, bg=BG)
        header.pack(fill="x", anchor="w")

        tk.Label(header, text=label.upper(), font=("Segoe UI", 8, "bold"),
                 bg=BG, fg=MUTED).pack(side="left", anchor="w")

        copy_btn = self._mk_btn(
            header, "📋 Copy", lambda v=value: self._copy_to_clipboard(v),
            BTN_BG, padx=6, pady=1, font=FONT_SMALL
        )
        copy_btn.pack(side="left", padx=(8, 0))

        def _flash_copied(b=copy_btn):
            orig = b.cget("text")
            b.config(text="✓ Copied")
            self.root.after(900, lambda: b.config(text=orig))
        copy_btn.config(command=lambda v=value, b=copy_btn: (self._copy_to_clipboard(v), _flash_copied(b)))

        # Use a Text widget so content is selectable and wraps properly
        txt = tk.Text(
            container,
            font=FONT_LABEL if key not in ("id", "chunk_id", "source_doc") else FONT_MONO,
            bg=PANEL, fg=accent_col,
            relief="flat", bd=0,
            wrap="word",
            padx=10, pady=8,
            cursor="xterm",
        )
        txt.insert("1.0", value)
        txt.config(state="disabled")

        # Auto-size height (max 12 lines)
        lines = value.count("\n") + 1
        avg_chars_per_line = 90
        wrapped = sum(max(1, len(line) // avg_chars_per_line + 1)
                      for line in value.splitlines()) if value else 1
        height = min(max(lines, wrapped, 1), 12)
        txt.config(height=height)
        txt.pack(fill="x", anchor="w")

        # Separator
        tk.Frame(container, bg=BORDER, height=1).pack(fill="x", pady=(6, 0))

    # ── Canvas / scroll helpers ───────────────────────────────────────────────

    def _on_frame_configure(self, _event=None):
        self.canvas.configure(scrollregion=self.canvas.bbox("all"))

    def _on_canvas_configure(self, event):
        self.canvas.itemconfig(self.canvas_window, width=event.width)

    # ── Status bar ────────────────────────────────────────────────────────────

    def _update_status(self):
        if not self.filepath:
            self.status_var.set("No file loaded.")
            return
        total   = len(self.records)
        labeled = sum(1 for r in self.records if "human_label" in r)
        pct     = f"{labeled/total*100:.1f}%" if total else "0%"
        self.status_var.set(
            f"{self.filepath}   |   {total} records   |   {labeled} labeled ({pct})"
            f"   |   Last saved: {datetime.now().strftime('%H:%M:%S')}"
        )


# ── Entry point ───────────────────────────────────────────────────────────────

def main():
    root = tk.Tk()
    root.geometry("1250x820")
    app = HallucinationLabeler(root)
    root.mainloop()


if __name__ == "__main__":
    main()

KeyboardInterrupt: 

: 